# Module 0, Notebook 3: Bellman Equations Lab

## Learning Objectives
By completing this notebook, you will:
1. Derive and implement the Bellman expectation equations for V and Q
2. Solve a small MDP exactly using the Bellman equations as a linear system
3. Compute V and Q for a given policy by iterative evaluation
4. Visualize value functions as heatmaps on a GridWorld
5. Verify that exact solutions match Monte Carlo estimates from Notebook 2

## Prerequisites
- Notebook 02: MDP representation, discount factor, returns
- Basic linear algebra: matrix-vector multiplication, system of equations
- NumPy array operations

## Estimated Time: 15 minutes

---

**Key Insight:** The Bellman equation is a fixed-point equation: $V = \mathcal{T}^\pi V$. This means the value function is a self-consistent assignment of values to states — the value of being in a state equals the expected immediate reward plus the discounted value of where you end up. Iterating this operator to convergence gives the true value function.

In [ ]:
learning_objectives([
    "Notebook 02: MDP representation, discount factor, returns",
    "Basic linear algebra: matrix-vector multiplication, system of equations",
    "NumPy array operations",
    "**Key Insight:** The Bellman equation is a fixed-point equation: $V = \mathcal{T}^\pi V$. This means the value function is a self-consistent assignment of values to states — the value of being in a state equals the expected immediate reward plus the discounted value of where you end up. Iterating this operator to convergence gives the true value function."
])

## 1. Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Imports and reproducibility
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

np.random.seed(42)
print("Ready. NumPy version:", np.__version__)

## 2. The Bellman Expectation Equations

For a fixed policy $\pi$, the **Bellman expectation equation** for the state value function $V^\pi$ is:

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)\bigl[R(s,a,s') + \gamma V^\pi(s')\bigr]$$

For the action-value function $Q^\pi$:

$$Q^\pi(s,a) = \sum_{s'} P(s'|s,a)\bigl[R(s,a,s') + \gamma \sum_{a'} \pi(a'|s') Q^\pi(s',a')\bigr]$$

The key relationships:
- $V^\pi(s) = \sum_a \pi(a|s) Q^\pi(s,a)$ — V is a weighted average of Q over actions
- $Q^\pi(s,a) = \sum_{s'} P(s'|s,a)[R + \gamma V^\pi(s')]$ — Q bootstraps from V

We'll implement three ways to compute $V^\pi$:
1. **Iterative policy evaluation** (repeated application of the Bellman operator)
2. **Direct linear solve** (treating the Bellman equations as $Ax = b$)
3. **Verification against Monte Carlo** from Notebook 2

## 3. Define the GridWorld MDP in Matrix Form

For the Bellman equations, we need transition matrices and reward vectors. We build a 4x4 GridWorld and convert it into dense transition matrices $P^\pi \in \mathbb{R}^{|S| \times |S|}$ and reward vector $R^\pi \in \mathbb{R}^{|S|}$ for a given policy $\pi$.

Once we have these, the Bellman equation becomes: $V^\pi = R^\pi + \gamma P^\pi V^\pi$, or equivalently $(I - \gamma P^\pi)V^\pi = R^\pi$.

In [ ]:
# Purpose: Define a 4x4 GridWorld as explicit transition matrices
# Key Concept: Matrix form makes the linear system structure of Bellman equations visible

# Grid layout: 0=empty, 1=hole, 2=goal, 3=start
GRID = np.array([
    [3, 0, 0, 0],
    [0, 1, 0, 1],
    [0, 0, 0, 1],
    [1, 0, 0, 2]
])

N_ROWS, N_COLS = GRID.shape
N_STATES  = N_ROWS * N_COLS   # 16
N_ACTIONS = 4                  # UP, DOWN, LEFT, RIGHT
ACTION_UP, ACTION_DOWN, ACTION_LEFT, ACTION_RIGHT = 0, 1, 2, 3

# Reward for each cell type
CELL_REWARDS = {0: -0.1, 1: -5.0, 2: 10.0, 3: -0.1}

# Terminal states: holes (1) and goal (2)
TERMINAL_STATES = set()
for r in range(N_ROWS):
    for c in range(N_COLS):
        if GRID[r, c] in (1, 2):
            TERMINAL_STATES.add(r * N_COLS + c)

def pos_to_state(r, c):
    return r * N_COLS + c

def state_to_pos(s):
    return s // N_COLS, s % N_COLS


def build_transition_and_reward(grid, n_states, n_actions):
    """
    Build dense transition tensor P[s, a, s'] and reward matrix R[s, a].

    This GridWorld is deterministic: each (state, action) maps to exactly one next state.
    Wall collisions keep the agent in place.

    Returns
    -------
    P : np.ndarray of shape (n_states, n_actions, n_states)
        P[s, a, s'] = probability of transitioning from s to s' via action a.
    R : np.ndarray of shape (n_states, n_actions)
        R[s, a] = expected immediate reward for (s, a).
    """
    P = np.zeros((n_states, n_actions, n_states))
    R = np.zeros((n_states, n_actions))

    deltas = {ACTION_UP: (-1, 0), ACTION_DOWN: (1, 0),
              ACTION_LEFT: (0, -1), ACTION_RIGHT: (0, 1)}

    for s in range(n_states):
        r, c = state_to_pos(s)

        # Terminal states are absorbing: all actions loop back to same state with 0 reward
        if s in TERMINAL_STATES:
            for a in range(n_actions):
                P[s, a, s] = 1.0
                R[s, a]    = 0.0
            continue

        for a in range(n_actions):
            dr, dc = deltas[a]
            nr, nc = r + dr, c + dc

            # Wall collision: stay in current cell
            if not (0 <= nr < N_ROWS and 0 <= nc < N_COLS):
                nr, nc = r, c

            s_next = pos_to_state(nr, nc)
            P[s, a, s_next] += 1.0
            R[s, a] = CELL_REWARDS[grid[nr, nc]]

    return P, R


P_tensor, R_matrix = build_transition_and_reward(GRID, N_STATES, N_ACTIONS)

# Sanity check
assert np.allclose(P_tensor.sum(axis=2), 1.0), "All rows of P must sum to 1"
print(f"Transition tensor P shape: {P_tensor.shape}")
print(f"Reward matrix R shape:     {R_matrix.shape}")
print(f"Terminal states (flat indices): {sorted(TERMINAL_STATES)}")

## 4. Method 1 — Iterative Policy Evaluation

Iterative policy evaluation repeatedly applies the Bellman operator until convergence:

$$V_{k+1}(s) \leftarrow \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)\bigl[R(s,a,s') + \gamma V_k(s')\bigr]$$

We stop when $\max_s |V_{k+1}(s) - V_k(s)| < \theta$ (a small tolerance). This converges because the Bellman operator is a contraction mapping with constant $\gamma$.

In [ ]:
# Purpose: Implement iterative policy evaluation
# Key Concept: Contraction mapping guarantees convergence regardless of starting V

def policy_evaluation_iterative(P_tensor, R_matrix, policy, gamma=0.99,
                                 theta=1e-8, max_iter=1000):
    """
    Compute V^pi by iteratively applying the Bellman expectation operator.

    Parameters
    ----------
    P_tensor : np.ndarray of shape (n_states, n_actions, n_states)
    R_matrix : np.ndarray of shape (n_states, n_actions)
    policy : np.ndarray of shape (n_states, n_actions)
        Stochastic policy: policy[s, a] = pi(a|s). Rows must sum to 1.
    gamma : float
    theta : float
        Convergence threshold on max absolute delta.
    max_iter : int

    Returns
    -------
    V : np.ndarray of shape (n_states,)
    history : list of float
        Max delta per iteration (for convergence visualization).
    """
    n_states = P_tensor.shape[0]
    V = np.zeros(n_states)
    history = []

    for iteration in range(max_iter):
        V_new = np.zeros(n_states)

        for s in range(n_states):
            # Bellman expectation update:
            # V_new(s) = sum_a pi(a|s) * sum_s' P(s'|s,a) * [R(s,a,s') + gamma * V(s')]
            # We use the expected reward R[s,a] and marginalize over s'
            for a in range(N_ACTIONS):
                # Expected next value under this action
                expected_next_V = np.dot(P_tensor[s, a], V)
                V_new[s] += policy[s, a] * (R_matrix[s, a] + gamma * expected_next_V)

        max_delta = np.max(np.abs(V_new - V))
        history.append(max_delta)
        V = V_new

        if max_delta < theta:
            print(f"Converged in {iteration + 1} iterations (max delta = {max_delta:.2e})")
            break

    return V, history


# Define a random policy (uniform over all actions)
uniform_policy = np.ones((N_STATES, N_ACTIONS)) / N_ACTIONS

GAMMA = 0.99
V_iter, delta_history = policy_evaluation_iterative(
    P_tensor, R_matrix, uniform_policy, gamma=GAMMA
)

print("\nValue function V (uniform random policy):")
print(V_iter.reshape(N_ROWS, N_COLS).round(3))

## 5. Method 2 — Exact Linear Solve

The Bellman equation $(I - \gamma P^\pi)V^\pi = R^\pi$ is a linear system. For small MDPs, we solve it directly using `np.linalg.solve`. This gives the exact answer in one step — no iteration required.

Here $P^\pi_{ss'} = \sum_a \pi(a|s) P(s'|s,a)$ is the policy-weighted transition matrix.

In [ ]:
# Purpose: Solve the Bellman linear system exactly
# Key Concept: For small MDPs, linear solve is faster and more accurate than iteration

def policy_evaluation_exact(P_tensor, R_matrix, policy, gamma=0.99):
    """
    Solve (I - gamma * P^pi) V = R^pi exactly via np.linalg.solve.

    Returns
    -------
    V : np.ndarray of shape (n_states,)
    """
    n_states = P_tensor.shape[0]

    # Step 1: Build the policy-weighted transition matrix P^pi (n_states x n_states)
    # P^pi[s, s'] = sum_a pi(a|s) * P(s'|s,a)
    P_pi = np.einsum('sa,sas->ss', policy, P_tensor)
    # Equivalently: P_pi = sum over a of policy[:, a:a+1] * P_tensor[:, a, :]

    # Step 2: Build the policy-weighted expected reward R^pi (n_states,)
    # R^pi[s] = sum_a pi(a|s) * R[s, a]
    R_pi = np.sum(policy * R_matrix, axis=1)

    # Step 3: Set up and solve (I - gamma * P^pi) V = R^pi
    A = np.eye(n_states) - gamma * P_pi
    V = np.linalg.solve(A, R_pi)

    return V


V_exact = policy_evaluation_exact(P_tensor, R_matrix, uniform_policy, gamma=GAMMA)

print("Exact V (linear solve):")
print(V_exact.reshape(N_ROWS, N_COLS).round(3))

max_diff = np.max(np.abs(V_iter - V_exact))
print(f"\nMax absolute difference between iterative and exact: {max_diff:.2e}")
print("Methods agree!" if max_diff < 1e-5 else "WARNING: large discrepancy detected!")

## 6. Computing the Q Function

Once we have $V^\pi$, the Q function follows directly from the Bellman relationship:

$$Q^\pi(s,a) = \sum_{s'} P(s'|s,a)\bigl[R(s,a,s') + \gamma V^\pi(s')\bigr]$$

Q gives us more information than V: it tells us the value of taking a specific action in a specific state, which is exactly what we need to improve a policy.

In [ ]:
# Purpose: Compute Q from V using the Bellman relationship
# Key Concept: Q(s,a) = immediate reward + discounted expected V of next state

def compute_Q_from_V(P_tensor, R_matrix, V, gamma=0.99):
    """
    Compute Q^pi from V^pi using the one-step Bellman equation.

    Parameters
    ----------
    P_tensor : np.ndarray (n_states, n_actions, n_states)
    R_matrix : np.ndarray (n_states, n_actions)
    V : np.ndarray (n_states,)
    gamma : float

    Returns
    -------
    Q : np.ndarray (n_states, n_actions)
    """
    # Q[s, a] = R[s, a] + gamma * sum_s' P[s, a, s'] * V[s']
    # This is equivalent to: Q = R + gamma * P_tensor @ V
    expected_next_V = np.einsum('sas,s->sa', P_tensor, V)  # shape (n_states, n_actions)
    Q = R_matrix + gamma * expected_next_V
    return Q


Q_uniform = compute_Q_from_V(P_tensor, R_matrix, V_exact, gamma=GAMMA)

print("Q function shape:", Q_uniform.shape)
print("\nQ[state, action] for first 4 states (UP, DOWN, LEFT, RIGHT):")
print("State  |    UP    DOWN    LEFT   RIGHT")
print("-" * 45)
for s in range(4):
    r, c = state_to_pos(s)
    q_vals = ", ".join(f"{q:7.3f}" for q in Q_uniform[s])
    print(f"({r},{c})   |  {q_vals}")

## 7. Visualizing Value Functions as Heatmaps

Heatmaps are the canonical way to visualize $V$ and $Q$ on a GridWorld. High values (warm colors) indicate states the agent should aim for; low values (cool colors) indicate states to avoid.

We also extract the greedy policy from Q: $\pi^*(s) = \arg\max_a Q(s,a)$, and display it as an arrow overlay.

In [ ]:
# Purpose: Visualize V as a heatmap and Q as per-action maps with policy arrows
# Key Concept: Spatial visualization of V reveals the value landscape the agent navigates

ACTION_ARROWS = {0: '↑', 1: '↓', 2: '←', 3: '→'}
GRID_SYMBOLS  = {0: '',  1: 'H', 2: 'G', 3: 'S'}

def plot_value_heatmap(V, Q, grid, title_prefix=''):
    """
    Plot V as heatmap and per-action Q maps.

    Parameters
    ----------
    V : np.ndarray (n_states,)
    Q : np.ndarray (n_states, n_actions)
    grid : np.ndarray (n_rows, n_cols)
    title_prefix : str
    """
    n_rows, n_cols = grid.shape
    n_states = n_rows * n_cols
    V_grid = V.reshape(n_rows, n_cols)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # --- Plot 1: V heatmap with policy arrows ---
    ax = axes[0]
    im = ax.imshow(V_grid, cmap='RdYlGn', interpolation='nearest', aspect='equal')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    greedy_actions = np.argmax(Q, axis=1)

    for r in range(n_rows):
        for c in range(n_cols):
            s = pos_to_state(r, c)
            v_val = V_grid[r, c]
            cell_sym = GRID_SYMBOLS.get(grid[r, c], '')

            # Value number
            ax.text(c, r, f'{v_val:.1f}', ha='center', va='top',
                    fontsize=8, color='black', fontweight='bold')
            # Cell label (S/H/G) and greedy action arrow
            if s not in TERMINAL_STATES:
                arrow = ACTION_ARROWS[greedy_actions[s]]
                ax.text(c, r + 0.35, arrow, ha='center', va='center',
                        fontsize=14, color='#1c7ed6', fontweight='bold')
            if cell_sym:
                ax.text(c, r - 0.35, cell_sym, ha='center', va='center',
                        fontsize=10, color='darkred', fontweight='bold')

    ax.set_title(f'{title_prefix}State Value Function V\n(arrows = greedy policy)',
                 fontsize=12, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

    # --- Plot 2: Q heatmap for best two actions ---
    for plot_idx, (action_idx, action_name) in enumerate(
        [(ACTION_DOWN, 'DOWN'), (ACTION_RIGHT, 'RIGHT')]
    ):
        ax = axes[plot_idx + 1]
        Q_grid = Q[:, action_idx].reshape(n_rows, n_cols)
        im = ax.imshow(Q_grid, cmap='RdYlGn', interpolation='nearest', aspect='equal')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        for r in range(n_rows):
            for c in range(n_cols):
                ax.text(c, r, f'{Q_grid[r, c]:.1f}', ha='center', va='center',
                        fontsize=9, color='black', fontweight='bold')

        ax.set_title(f'{title_prefix}Q(s, {action_name})', fontsize=12, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.savefig(f'value_functions_{title_prefix.lower().replace(" ","_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()


plot_value_heatmap(V_exact, Q_uniform, GRID, title_prefix='Uniform Policy: ')

## 8. Convergence Visualization

The convergence history from iterative policy evaluation tells us how quickly the Bellman operator contracts. Plotting $\max_s |V_{k+1}(s) - V_k(s)|$ on a log scale should show a straight line — exponential convergence at rate $\gamma$.

In [ ]:
# Purpose: Plot the convergence curve of iterative policy evaluation
# Key Concept: Log-linear decay confirms the contraction rate = gamma

fig, ax = plt.subplots(figsize=(9, 4))

ax.semilogy(delta_history, color='#1c7ed6', linewidth=2)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Max $|\\Delta V|$ (log scale)', fontsize=12)
ax.set_title(f'Convergence of Iterative Policy Evaluation ($\\gamma={GAMMA}$)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')

# Annotate convergence iteration
ax.axhline(1e-8, color='red', linestyle='--', linewidth=1, label='Threshold $\\theta=10^{-8}$')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('bellman_convergence.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Converged in {len(delta_history)} iterations")

## 9. Exercises

### Exercise 3.1: Compute V for a Deterministic Policy

**Task:** Define a deterministic policy that always moves RIGHT (action 3) from every state. Represent it as a stochastic policy matrix of shape (16, 4) — all probability mass on action 3. Compute $V^\pi$ using both iterative evaluation and exact linear solve. Verify they agree within $10^{-5}$.

**Expected Output:** `V_right_iter` and `V_right_exact` printed as 4x4 grids, and a confirmation that they match.

**Hints:**
<details>
<summary>Hint 1</summary>
A deterministic policy that always takes action 3 (RIGHT) can be written as a matrix where column 3 is all 1.0 and all other columns are 0.0.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
```python
right_policy = np.zeros((N_STATES, N_ACTIONS))
right_policy[:, ACTION_RIGHT] = 1.0
```
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Step 1: Define the always-RIGHT policy matrix
right_policy = None  # shape (N_STATES, N_ACTIONS)

# Step 2: Compute V using iterative evaluation
V_right_iter  = None
delta_right   = None

# Step 3: Compute V using exact linear solve
V_right_exact = None

# Step 4: Check agreement
max_diff_right = None

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_1():
    assert right_policy is not None, "Define right_policy!"
    assert right_policy.shape == (N_STATES, N_ACTIONS), \
        f"right_policy must have shape ({N_STATES}, {N_ACTIONS})"
    assert np.allclose(right_policy.sum(axis=1), 1.0), \
        "Each row of right_policy must sum to 1 (valid probability distribution)."
    assert np.all(right_policy[:, ACTION_RIGHT] == 1.0) or \
           np.all(right_policy[0, ACTION_RIGHT] == 1.0), \
        "Policy should assign probability 1 to action RIGHT."

    assert V_right_iter  is not None, "Run policy_evaluation_iterative to get V_right_iter."
    assert V_right_exact is not None, "Run policy_evaluation_exact to get V_right_exact."
    assert max_diff_right is not None, "Compute max_diff_right!"

    assert V_right_iter.shape  == (N_STATES,), f"V_right_iter shape should be ({N_STATES},)"
    assert V_right_exact.shape == (N_STATES,), f"V_right_exact shape should be ({N_STATES},)"
    assert max_diff_right < 1e-4, \
        f"Iterative and exact solutions differ by {max_diff_right:.2e} (should be < 1e-4)"

    print(f"Exercise 3.1 passed! Max diff between methods: {max_diff_right:.2e}")
    print("V (always-RIGHT policy):")
    print(V_right_exact.reshape(N_ROWS, N_COLS).round(2))

test_exercise_3_1()

### Exercise 3.2: Extract the Greedy Policy and Compute Its Value

**Task:** Given `V_exact` (value function of the uniform random policy), extract the greedy policy $\pi^{greedy}(s) = \arg\max_a Q(s, a)$. Convert it to a stochastic policy matrix. Compute $V^{\pi^{greedy}}$ using the exact method. Compare $V^{\pi^{greedy}}$ to $V^{uniform}$ at state 0 (top-left corner). Is the greedy policy better?

**Expected Output:** Comparison of V[0] under both policies.

**Hints:**
<details>
<summary>Hint 1</summary>
Use `np.argmax(Q_uniform, axis=1)` to get the greedy action for each state, then set `greedy_policy_matrix[s, best_action] = 1.0`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Step 1: Extract greedy action per state from Q_uniform
greedy_actions_per_state = None  # shape (N_STATES,)

# Step 2: Build stochastic policy matrix for the greedy policy
greedy_policy_matrix = None  # shape (N_STATES, N_ACTIONS)

# Step 3: Compute V for the greedy policy
V_greedy = None

# Step 4: Compare V[0] under both policies
v0_uniform = None
v0_greedy  = None

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_2():
    assert greedy_actions_per_state is not None, "Compute greedy_actions_per_state!"
    assert greedy_actions_per_state.shape == (N_STATES,), \
        f"Expected shape ({N_STATES},), got {greedy_actions_per_state.shape}"
    assert all(a in range(N_ACTIONS) for a in greedy_actions_per_state), \
        "All actions must be in [0, N_ACTIONS-1]."

    assert greedy_policy_matrix is not None, "Build greedy_policy_matrix!"
    assert greedy_policy_matrix.shape == (N_STATES, N_ACTIONS)
    assert np.allclose(greedy_policy_matrix.sum(axis=1), 1.0), \
        "Each row must sum to 1."

    assert V_greedy is not None, "Compute V_greedy using policy_evaluation_exact!"
    assert v0_uniform is not None and v0_greedy is not None

    print(f"Exercise 3.2 passed!")
    print(f"  V[0] under uniform policy: {v0_uniform:.4f}")
    print(f"  V[0] under greedy policy:  {v0_greedy:.4f}")
    better = 'Greedy' if v0_greedy > v0_uniform else 'Uniform'
    print(f"  Better at state 0: {better} policy")
    print("  Note: greedy w.r.t. uniform's Q may not be optimal — that requires Policy/Value Iteration (Module 1).")

test_exercise_3_2()

In [ ]:
section_divider("Key Takeaways")

## Key Takeaways

1. **The Bellman equation is a fixed-point equation**: $V^\pi = \mathcal{T}^\pi V^\pi$. Iterating to convergence gives the true value function for any policy $\pi$.
2. **Two equivalent methods**: iterative evaluation (apply the Bellman operator repeatedly until $\Delta V < \theta$) and direct linear solve ($V = (I - \gamma P^\pi)^{-1} R^\pi$). Both give the same answer; iteration scales better.
3. **Q connects to V via one Bellman backup**: $Q^\pi(s,a) = \sum_{s'} P(s'|s,a)[R + \gamma V^\pi(s')]$. You only need to solve for V.
4. **Convergence is exponential** at rate $\gamma$. Smaller $\gamma$ means faster convergence but a shorter planning horizon.
5. **Heatmaps reveal the value landscape**: states near the goal have high V; holes and dead ends have low V.
6. **Greedy policy improvement**: taking $\arg\max_a Q(s,a)$ at every state is guaranteed to produce a policy at least as good as the current one. This is the foundation of policy iteration.

## What's Next

In Module 1, Notebook 1 (`01_policy_evaluation.ipynb`), we build the full policy evaluation algorithm for the 4x4 GridWorld and track convergence sweep by sweep. Then Notebook 2 combines evaluation with improvement into **policy iteration** and **value iteration** — the two classical exact RL algorithms.

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])